# Data Cleaning
**Phase 1: Counting**  
Objective: Quantify the "noise" and structure of the raw text.

[x] `emoji_count`: Create a new column to store the count of emojis per comment (using Unicode regex).

[x] `punct_count`: Create a new column to count standard punctuation (., ,, !, ?, ;, :) to evaluate sentence structure and readability.

**Phase 2: Lighter Cleaning (LLM-Ready)**  
Objective: Prepare text for sentiment analysis while preserving context.

[x] Emoji Transformation: Use emoji.demojize() to convert raw emojis into text descriptors (e.g., 😊 → :smiling_face:). This allows LLMs to "read" the emotion without losing meaning.

[x] Context Preservation: Retain original punctuation and stopwords, as these are critical for maintaining the nuance and "voice" of the community.

**Phase 3: Heavier Cleaning (NLP-Ready)**  
Objective: Normalize data for traditional linguistic and statistical modeling.

[x] Tokenization & Normalization: Strip non-alphanumeric characters.

[x] Stopword Removal: Filter out high-frequency, low-value words (e.g., "the", "and", "is").

[x] Lowercasing: Standardize all text to improve word frequency accuracy.

**Phase 4: Targeted Windowing**  
Objective: Create manageable input segments for Islander-specific sentiment analysis. These windows must be large enough to understand sentiment and polarization. We should also be removing the islander (nick)names.

[ ] Islander Entity Mapping: Compile a dictionary of islander names/nicknames to filter windows—ignore windows that don't reference a target.

[ ] Sliding Windows: Segment long comments into 50–75 word chunks with a 25-word overlap to ensure context preservation.

[ ] Target Mapping: Filter windows by the presence of specific Islander names to create highly focused sentiment datasets.

**Shortcomings**  
I will NOT be looking at the essence of replies. That is, maybe there could be a comment that created more polarized debate. I do not wish to do this.

## Setup

### Libraries

In [1]:
import duckdb
import pandas as pd

import re
import contractions
import emoji
import html
import ftfy


### Stopwords

In [2]:
import nltk
from nltk.corpus import stopwords

# Download standard list of English stopwords
nltk.download('stopwords')

# 1. Start with standard NLTK
custom_stopwords = set(stopwords.words('english'))

# 2. Add your Reddit/Internet-specific "filler"
# These are words that appear constantly but carry no sentiment value 
# regarding the Islanders themselves.
internet_fillers = {
    'lol', 'lmao', 'lmfao', 'fr', 'rn', 'ig', 'idk', 'im', 'u', 'ur', 
    'n', 'gon', 'wanna', 'gonna', 'literally', 'actually', 'totally',
    'just', 'like', 'really', 'would', 'could', 'think', 'say', 'know'
}

# 3. Combine them
all_stopwords = custom_stopwords.union(internet_fillers)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ehze\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


We need to double-check that we are in the right folder.

In [3]:
import os
# Print the current working directory so you know where you are
print("Current folder:", os.getcwd())

# List all files in the directory one level up to find where your DB really is
import os
print("Files in parent folder:", os.listdir("../data/"))

Current folder: c:\Users\ehze\Desktop\data-by-edie\004-love-island\notebooks
Files in parent folder: ['LoveIslandUSA_S7', 'love_island_reddit.duckdb']


Let's see what's in our database.

In [4]:
con = duckdb.connect("../data/love_island_reddit.duckdb")

# This will print the actual table names to your screen
print(con.execute("SHOW TABLES").df())

con.close()

               name
0          comments
1  islander_windows
2       submissions
3             users


### Helpers

In [5]:
# Emoji count helper
def get_emoji_count(text):
    if not isinstance(text, str): return 0
    return len([char for char in text if char in emoji.EMOJI_DATA])

# Punctuation count helper
def get_punct_count(text):
    if not isinstance(text, str): return 0
    return len(re.findall(r'[.,!?;:]', text))

# Light cleaning: just remove emojis
def light_clean(text):
    if not isinstance(text, str): return text
    return emoji.demojize(text)

# Text to string, handling None and NaN values
def safe_text(text):

    if text is None:
        return ""

    try:
        if pd.isna(text):
            return ""
    except Exception:
        pass

    return str(text)

# Remove URLs from text
def remove_urls(text):
    return re.sub(r"http\S+|www\S+", " ", text, flags=re.IGNORECASE)

# Normalize text: unescape HTML, fix text, expand contractions, remove URLs
def normalize_all(text):
    # This combines your normalization and safe_text
    text = safe_text(text)
    text = html.unescape(text)
    text = ftfy.fix_text(text)
    text = contractions.fix(text)
    text = remove_urls(text)
    return text

# Clean text lightly: normalize and remove extra whitespace
def clean_light(text):
    text = normalize_all(text)
    return re.sub(r"\s+", " ", text).strip()

# Clean text heavily: normalize, convert emojis to tags, and strip non-alphanumeric characters
def clean_heavy(text):
    text = normalize_all(text)
    text = emoji.demojize(text)
    # Convert emojis to tags
    text = re.sub(r":([a-zA-Z0-9_+-]+):", r" [emoji_\1] ", text)
    # Lowercase and remove non-alphanumerics
    text = re.sub(r"[^a-z0-9\s'\[\]_]", " ", text.lower())
    
    # Tokenize and remove stopwords
    words = text.split()
    filtered_words = [w for w in words if w not in all_stopwords]
    
    return " ".join(filtered_words).strip()

    

## Apply

In [6]:
# 1. Connect
con = duckdb.connect("../data/love_island_reddit.duckdb")

# 2. Add all columns at once
con.execute("""
    ALTER TABLE comments ADD COLUMN IF NOT EXISTS emoji_count INTEGER DEFAULT 0;
    ALTER TABLE comments ADD COLUMN IF NOT EXISTS punct_count INTEGER DEFAULT 0;
    ALTER TABLE comments ADD COLUMN IF NOT EXISTS body_clean_light TEXT;
    ALTER TABLE comments ADD COLUMN IF NOT EXISTS body_clean_heavy TEXT;
""")

# 3. Pull data
df = con.execute("SELECT id, body FROM comments").df()

# 4. Apply everything (using your modular utilities)
df['emoji_count'] = df['body'].apply(get_emoji_count)
df['punct_count'] = df['body'].apply(get_punct_count)
df['body_clean_light'] = df['body'].apply(clean_light)
df['body_clean_heavy'] = df['body'].apply(clean_heavy)

# 5. Push everything back to DuckDB
con.execute("CREATE OR REPLACE TEMP TABLE updates AS SELECT id, emoji_count, punct_count, body_clean_light, body_clean_heavy FROM df")

con.execute("""
    UPDATE comments 
    SET emoji_count = updates.emoji_count,
        punct_count = updates.punct_count,
        body_clean_light = updates.body_clean_light,
        body_clean_heavy = updates.body_clean_heavy
    FROM updates 
    WHERE comments.id = updates.id
""")

print("Transformation complete: Light, Heavy, and Metrics added.")
con.close()

Transformation complete: Light, Heavy, and Metrics added.


## Quality Control

In [7]:
con = duckdb.connect("../data/love_island_reddit.duckdb")

# 1. Descriptive stats for the new columns
summary = con.execute("""
    SELECT 
        COUNT(*) as total_comments,
        AVG(emoji_count) as avg_emojis,
        MAX(emoji_count) as max_emojis,
        AVG(punct_count) as avg_punct,
        MAX(punct_count) as max_punct,
        -- How many people don't use emojis/punct at all?
        SUM(CASE WHEN emoji_count = 0 THEN 1 ELSE 0 END) as zero_emoji_count,
        SUM(CASE WHEN punct_count = 0 THEN 1 ELSE 0 END) as zero_punct_count
    FROM comments
""").df()

print("--- Feature Summary Statistics ---")
print(summary.T) # .T transposes it so it's easier to read vertically

# 2. Distribution check (The "Is this normal?" check)
# See the top 10 most emoji-heavy comments
print("\n--- Top 10 Emoji-Heavy Comments ---")
print(con.execute("SELECT body, emoji_count FROM comments ORDER BY emoji_count DESC LIMIT 10").df())

con.close()

--- Feature Summary Statistics ---
                             0
total_comments    12130.000000
avg_emojis            0.257296
max_emojis           18.000000
avg_punct             2.165705
max_punct            57.000000
zero_emoji_count  10279.000000
zero_punct_count   3852.000000

--- Top 10 Emoji-Heavy Comments ---
                                                body  emoji_count
0  Nothing but psychics in this sub 🔮🧙‍♀️🧙‍♂️\n\n...           18
1                                   😂😂😂😂😂😭😭😭👏🏾👏🏾👏🏾👏🏾           16
2  A lot to uncover here let’s see how I do 🤣 Aus...           13
3                     Insane chemistry 💀💀💀🤡🤡🤮🤣🤣🤣🤣🤣🤣🤣           13
4  Emotional intelligence is so underrated, but C...           12
5  It’s just as annoying as the Huda hoppers 🙄🙄🙄🙄...           10
6  NICOLANDRIA NATION WE’RE COMING HOME 🇺🇸🇺🇸🇺🇸🇺🇸🇺...            9
7                                           😂😂😂😂😂😂😂💀            8
8                     fr i’m still smiling😩😩😩🙂‍↔️😭😆😍            8
9                   

In [8]:
import duckdb

con = duckdb.connect("../data/love_island_reddit.duckdb")

# Preview the raw, light, and heavy bodies
print("--- Top 3 Rows Inspection ---")
print(con.execute("SELECT body, body_clean_light, body_clean_heavy FROM comments LIMIT 3").df())

con.close()

--- Top 3 Rows Inspection ---
                                                body  \
0  Why would we ask any contestant who goes throu...   
1  Wouldn’t choosing true love even after all the...   
2  That's how it's done in every single LIUSA fin...   

                                    body_clean_light  \
0  Why would we ask any contestant who goes throu...   
1  Would not choosing true love even after all th...   
2  That is how it is done in every single LIUSA f...   

                                    body_clean_heavy  
0  ask contestant goes scrutiny drama inside vill...  
1  choosing true love even went prove even love m...  
2  done every single liusa finale ever aired choo...  


Here, we verify whether we have cleaned up the data.

In [9]:
con = duckdb.connect("../data/love_island_reddit.duckdb")

# Calculate average length (in characters)
stats = con.execute("""
    SELECT 
        AVG(LENGTH(body)) as avg_raw_len,
        AVG(LENGTH(body_clean_light)) as avg_light_len,
        AVG(LENGTH(body_clean_heavy)) as avg_heavy_len
    FROM comments
""").df()

print("\n--- Average Character Length Comparison ---")
print(stats.T)

con.close()


--- Average Character Length Comparison ---
                        0
avg_raw_len    130.365128
avg_light_len  128.415169
avg_heavy_len   76.851937


I actually find no issue with this filter. If we have a `body_clean_heavy` that is too short, I wouldn't necessarily want to use that data anyway.

In [10]:
# Check for "Over-cleans"
# If a comment is now less than 10 characters, it might be purely emojis or empty.
con = duckdb.connect("../data/love_island_reddit.duckdb")

print("--- Potential Over-cleaned Comments ---")
print(con.execute("""
    SELECT body, body_clean_heavy 
    FROM comments 
    WHERE LENGTH(body_clean_heavy) < 10 
    LIMIT 10
""").df())

con.close()

--- Potential Over-cleaned Comments ---
                                                body body_clean_heavy
0                               Same, what was it???                 
1                              I was about to say...                 
2  https://preview.redd.it/esrn4q8aeu4f1.jpeg?wid...        got queen
3                                       Lost my MIND        lost mind
4  https://open.spotify.com/track/6mhSmJkz9ZayhIb...                 
5                                               Same                 
6                                               Same                 
7                               Looking for this too          looking
8  Found it! https://youtu.be/joJQNW0V4ik?si=Tf7j...            found
9  https://open.spotify.com/playlist/50L5FUSAxBlA...                 


## Season 7 Cast

In [ ]:
islander_map = {
    "Ace Greene": ["ace"],
    "Amaya Espinal": ["amaya", "amaya papaya", "papaya"],
    "Andreina Santos": ["andreina", "andreena", "adriena"],
    "Austin Shepard": ["austin"],
    "Bryan Arenales": ["bryan", "brian"],
    "Charlie Georgiou": ["charlie"],
    "Chris Seeley": ["chris"],
    "Cierra Ortega": ["cierra", "ciara", "sierra", "ciarra"],
    "Clarke Carraway": ["clarke", "clark"],
    "Courtney Watson": ["coco", "courtney"],
    "Elan Bibas": ["elan", "elon"],
    "Gracyn Blackmore": ["gracyn", "gracelyn", "graceyn"],
    "Hannah Fields": ["hannah", "hana"],
    "Huda Mustafa": ["huda", "hudabubbaaa", "hurricane huda"],
    "Iris Kendall": ["iris"],
    "Isabelle-Anne Walker": ["belle-a", "bellea", "isabelle", "belle", "bella"],
    "JD Dodard": ["jd"],
    "Jaden Duggar": ["jaden", "jaiden"],
    "Jalen Brown": ["jalen"],
    "Jeremiah Brown": ["jeremiah", "jeramiah"],
    "Jose Garcia": ["pepe", "jose"],
    "Michelle Bissainthe": ["chelley", "michelle", "cheley"],
    "Nicolas Vansteenberghe": ["nic", "nicolas", "nick", "big belgian", "nicolandria"],
    "Olandria Carthen": ["olandria", "ola", "nicolandria"],
    "Savanna Einerson": ["vanna", "savanna", "savannah"],
    "Taylor Williams": ["taylor"],
    "Thomas Palma": ["tj", "thomas"],
    "Yulissa Escobar": ["yulissa"],
    "Zac Woodworth": ["zac"],
    "Zak Srakaew": ["zak"]
}

## Create Target Windows

In [12]:
import duckdb
import re
import pandas as pd

# 1. Setup Master Pattern
all_names = []
for nicknames in islander_map.values():
    all_names.extend(nicknames)
all_names = sorted(list(set(all_names)), key=len, reverse=True)
name_pattern = re.compile(r'\b(' + '|'.join(map(re.escape, all_names)) + r')\b', re.IGNORECASE)

def get_word_windows(text, target_islander, window_size=10):
    words = text.split()
    # Pattern for this specific islander (canonical + nicknames)
    target_pattern = re.compile(r'\b(' + '|'.join(map(re.escape, islander_map[target_islander])) + r')\b', re.IGNORECASE)
    
    windows = []
    for i, word in enumerate(words):
        if target_pattern.match(word):
            start = max(0, i - window_size)
            end = min(len(words), i + window_size + 1)
            
            # Extract and anonymize
            window_words = words[start:end]
            clean_window = [name_pattern.sub("[ISLANDER]", w) for w in window_words]
            windows.append(" ".join(clean_window))
    return windows

# 2. Execution
con = duckdb.connect("../data/love_island_reddit.duckdb")
df = con.execute("SELECT id, body_clean_heavy FROM comments").df()

# Build the Expanded DataFrame
all_rows = []
for _, row in df.iterrows():
    # Check mentions for every Islander in the map
    for islander in islander_map.keys():
        windows = get_word_windows(row['body_clean_heavy'], islander, window_size=7)
        for win in windows:
            all_rows.append({
                'comment_id': row['id'],
                'target_islander': islander,
                'window_text': win
            })

df_windows = pd.DataFrame(all_rows)

# 3. Store the result
con.execute("CREATE OR REPLACE TABLE islander_windows AS SELECT * FROM df_windows")
print(f"Extracted {len(df_windows)} windows across all cast members.")
con.close()

Extracted 11829 windows across all cast members.


In [13]:
df_windows

,comment_id,target_islander,window_text
0,n8vj28f,Chris Seeley,still lucrative 100k unless course couple [ISL...
1,n8vj28f,Huda Mustafa,brand still lucrative 100k unless course coupl...
2,mvw5ffk,Olandria Carthen,[ISLANDER] open mouth tongue kisses straight a...
3,mvw8d1o,Michelle Bissainthe,love [ISLANDER]'s green bikini
4,mvw0lnd,Yulissa Escobar,[ISLANDER]'s appropriative ass outfit sgahebsh...
...,...,...,...
11824,n31mdrl,Chris Seeley,longer bound romance [ISLANDER] owed nothing [...
11825,n31mdrl,Huda Mustafa,longer bound romance [ISLANDER] owed nothing [...
11826,n31mdrl,Huda Mustafa,owed nothing [ISLANDER] helping hand extended ...
11827,n31mdrl,Huda Mustafa,water managed cross high time stop infantilizi...


In [14]:
con = duckdb.connect("../data/love_island_reddit.duckdb")

# Calculate counts and proportions in one go
proportion_df = con.execute("""
    SELECT 
        target_islander,
        COUNT(*) as mention_count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) as proportion_pct
    FROM islander_windows
    GROUP BY target_islander
    ORDER BY mention_count DESC
""").df()

con.close()

In [15]:
proportion_df

,target_islander,mention_count,proportion_pct
0,Huda Mustafa,1402,11.85
1,Olandria Carthen,1226,10.36
2,Nicolas Vansteenberghe,1212,10.25
3,Amaya Espinal,961,8.12
4,Ace Greene,882,7.46
5,Cierra Ortega,764,6.46
6,Taylor Williams,678,5.73
7,Michelle Bissainthe,586,4.95
8,Jeremiah Brown,510,4.31
9,Jose Garcia,496,4.19


## Check Absolute Target Counts
**Data Limitation**  
I was extremely skeptical of the number of windows created across cast members. Therefore, I checked how many comments were free of islander names. I have a large amount of S7 discussion that does not talk directly about the contestants, but rather about the logistics. That is because of the threads that I did choose to include.

I could also check the sentiments of comments without any targets just to check the sentiment, perhaps even compare as a sensitivity analysis.

In [16]:
import duckdb

con = duckdb.connect("../data/love_island_reddit.duckdb")

# Create a list of all names to search
all_names = []
for nicknames in islander_map.values():
    all_names.extend(nicknames)
all_names = sorted(list(set(all_names)), key=len, reverse=True)

# Count how many comments have at least one mention
# This tells you the "Density of Mentions" in your dataset
# Join the names with pipe |
regex_pattern = '|'.join(map(re.escape, all_names))

# Use regexp_matches() instead of REGEXP
stats = con.execute(f"""
    SELECT 
        COUNT(*) as total_comments,
        COUNT(*) FILTER (regexp_matches(body_clean_heavy, '({regex_pattern})')) as comments_with_mentions
    FROM comments
""").df()

print(stats)

con.close()

   total_comments  comments_with_mentions
0           12130                    6245


In [17]:
import duckdb
import re

# Rebuild your regex pattern
regex_pattern = '|'.join(map(re.escape, all_names))

con = duckdb.connect("../data/love_island_reddit.duckdb")

# Use NOT regexp_matches to find the "orphaned" comments
orphaned_comments = con.execute(f"""
    SELECT id, body_clean_heavy 
    FROM comments 
    WHERE NOT regexp_matches(body_clean_heavy, '{regex_pattern}')
    LIMIT 50
""").df()

con.close()

In [18]:
orphaned_comments.head(20)

,id,body_clean_heavy
0,n8wrbtw,choosing true love even went prove even love m...
1,n8vtokm,done every single liusa finale ever aired choo...
2,n8wmc46,guess hahaha winced fact supposed okey get
3,n8wo4ww,yeah honestly price pool feels forced win atte...
4,n8vqtm3,stop people choosing money connection continui...
5,n8wo4yf,nothing smart op good alternative ending
6,n8wsz11,absolutely nothing mainly judging notion prove...
7,n8vpcce,share steal choice done winners across seasons...
8,n8wljpp,also true popularity fame opportunities shadow...
9,n8wlkfv,cannot make
